# Credit Card Fraud Detection - Model Training and Evaluation

## Project Objective

The goal of this stage of the project is to train and evaluate machine learning models capable of detecting fraudulent credit card transactions. Because fraud detection datasets are typically highly imbalanced, special care must be taken when designing the training pipeline and selecting evaluation metrics.

This notebook focuses on:

- Training several classification algorithms
- Evaluating them using appropriate metrics for imabalenced data
- Testing different resampling strategies
- Selecting the best performing model
- Saving the trained model for future deployment

## Import libraries

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (average_precision_score, precision_recall_curve, confusion_matrix, roc_auc_score, 
                             precision_score, recall_score, f1_score)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

## Load dataset

In [2]:
data_path = "../data/processed/creditcard.csv"
base = pd.read_csv(data_path)

print("Dataset shape:", base.shape)
base.head()

Dataset shape: (283726, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Train / Validation / Test split

In [3]:
random_state = 20260219

In [4]:
X = base.drop(columns=["Class"])
y = base["Class"].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=random_state)

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=random_state)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (181584, 30)
Validation set: (45396, 30)
Test set: (56746, 30)


The dataset is split into three subsets:

- The training set: used to train the models
- The validation set: used to select the best classification threshold
- Test set: used for final performance

## Model definitions

In [5]:
def make_models():
    models = {
        "Logistic_Regression": LogisticRegression(max_iter=2000),
        "Random_Forest": RandomForestClassifier(n_estimators=200),
        "Naives_Bayes": GaussianNB()
    }
    
    return models

## Sampling strategies

Because the dataset is extremely imbalanced, several sampling strategies are evaluated:

• **Baseline** — no resampling  
• **SMOTE** — synthetic oversampling of fraud cases  
• **Undersampling** — reduction of majority class samples

These techniques aim to improve the model's ability to detect rare fraud events.

In [6]:
def build_pipeline(clf, strategy="baseline"):
    steps = []
    
    # Most models benefit from scaling. RF doesn't need it but doesn't hurt much here.
    steps.append(("scaler", StandardScaler()))
    
    if strategy == "smote":
        # Raise minority to 10% of majority to keep it realistic + avoid huge runtime
        steps.append(("sampler", SMOTE(random_state=random_state, sampling_strategy=0.1)))
    elif strategy == "under":
        # Keep minority fully, reduce majority (minority becomes 10% of majority)
        steps.append(("sampler", RandomUnderSampler(random_state=random_state, sampling_strategy=0.1)))
    
    steps.append(("clf", clf))
    
    return Pipeline(steps=steps)

## Threshold optimization

In [7]:
def pick_threshold_max_f1(y_true, y_proba):

    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)

    f1 = (2 * precision * recall) / np.clip(precision + recall, 1e-22, None)

    best_idx = int(np.nanargmax(f1[:-1]))

    return float(thresholds[best_idx])

## Model evaluation

In [8]:
def evaluate(model, X_val, y_val, X_test, y_test):

    val_proba = model.predict_proba(X_val)[:,1]

    threshold = pick_threshold_max_f1(y_val, val_proba)

    test_proba = model.predict_proba(X_test)[:,1]

    y_pred = (test_proba >= threshold).astype(int)

    metrics = {
        "AUPRC": average_precision_score(y_test, test_proba),
        "ROC_AUC": roc_auc_score(y_test, test_proba),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Threshold": threshold,
        "CM_TN": confusion_matrix(y_test, y_pred)[0, 0],
        "CM_FP": confusion_matrix(y_test, y_pred)[0, 1],
        "CM_FN": confusion_matrix(y_test, y_pred)[1, 0],
        "CM_TP": confusion_matrix(y_test, y_pred)[1, 1]
    }

    return metrics

## Main core

In [9]:
models = make_models()
strategies = ["baseline", "smote", "under"]

results = []

# Track best model
best_auprc = 0
best_model = None
best_model_name = None
best_strategy = None

for strat in strategies:
    print(f"\n======================================")
    print(f"Strategy: {strat.upper()}")
    print(f"\n======================================")
    
    for name, clf in models.items():
        pipe = build_pipeline(clf, strategy=strat)
        pipe.fit(X_train, y_train)
        
        metrics = evaluate(pipe, X_val, y_val, X_test, y_test)
        metrics["Model"] = name
        metrics["Strategy"] = strat
        results.append(metrics)
        
        print(
            f"{name:12s} | AUPRC={metrics['AUPRC']:.6f} "
            f"| P={metrics['Precision']:.4f} R={metrics['Recall']:.4f} "
            f"| F1={metrics['F1']:.4f} | thr={metrics['Threshold']:.6f}"
        )
        
        # Update best model
        if metrics["AUPRC"] > best_auprc:
            best_auprc = metrics["AUPRC"]
            best_model = pipe
            best_model_name = name
            best_strategy = strat

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["AUPRC"], ascending=False).reset_index(drop=True)

print("\n\n===== FINAL RANKING (by AUPRC) =====")
print(results_df[["Strategy", "Model", "AUPRC", "ROC_AUC", "Precision", "Recall", "F1", "Threshold"]].head(15))

results_df.to_csv("../results/model_comparison_results.csv", index=False)
print("\nSaved: model_comparison_results.csv")


Strategy: BASELINE

Logistic_Regression | AUPRC=0.763118 | P=0.7849 R=0.7684 | F1=0.7766 | thr=0.096842
Random_Forest | AUPRC=0.868078 | P=0.9250 R=0.7789 | F1=0.8457 | thr=0.485000
Naives_Bayes | AUPRC=0.092209 | P=0.1039 R=0.7789 | F1=0.1834 | thr=1.000000

Strategy: SMOTE

Logistic_Regression | AUPRC=0.765069 | P=0.7935 R=0.7684 | F1=0.7807 | thr=0.949616
Random_Forest | AUPRC=0.868279 | P=0.9186 R=0.8316 | F1=0.8729 | thr=0.485000
Naives_Bayes | AUPRC=0.097004 | P=0.1099 R=0.7684 | F1=0.1924 | thr=1.000000

Strategy: UNDER

Logistic_Regression | AUPRC=0.684932 | P=0.6822 R=0.7684 | F1=0.7228 | thr=0.961309
Random_Forest | AUPRC=0.849415 | P=0.8706 R=0.7789 | F1=0.8222 | thr=0.875000
Naives_Bayes | AUPRC=0.091863 | P=0.1041 R=0.7789 | F1=0.1836 | thr=1.000000


===== FINAL RANKING (by AUPRC) =====
   Strategy                Model     AUPRC   ROC_AUC  Precision    Recall  \
0     smote        Random_Forest  0.868279  0.977321   0.918605  0.831579   
1  baseline        Random_Forest 

## Save best model

In [10]:
# Save best model and metadata
if best_model is not None:
    
    model_filename = f"../models/best_model_{best_model_name}_{best_strategy}.joblib"
    metadata_filename = f"../models/model_metadata.json"
    
    # Save model
    joblib.dump(best_model, model_filename)
    
    # Metadata for production
    metadata = {
        "model_name": best_model_name,
        "sampling_strategy": best_strategy,
        "threshold": float(results_df.iloc[0]["Threshold"]),
        "features": list(X.columns),
        "training_date": datetime.now(timezone.utc).isoformat(),
        "metrics": {
            "AUPRC": float(results_df.iloc[0]["AUPRC"]),
            "ROC_AUC": float(results_df.iloc[0]["ROC_AUC"]),
            "Precision": float(results_df.iloc[0]["Precision"]),
            "Recall": float(results_df.iloc[0]["Recall"]),
            "F1": float(results_df.iloc[0]["F1"])
        }
    }
    
    # Save metadata
    with open(metadata_filename, "w") as f:
        json.dump(metadata, f, indent=4)
        
print(f"\nBest model saved as: {model_filename}")
print(f"Metadata saved as: {metadata_filename}")


Best model saved as: ../models/best_model_Random_Forest_smote.joblib
Metadata saved as: ../models/model_metadata.json


The best performing model is saved using `joblib`.  
This serialized model can later be used for inference or deployment in an API.

## Isolation Forest (Unsupervised Anomaly Detection)

Isolation Forest is an unsupervised algorithm that detects anomalies by isolating observations. Unlike the supervised models above, it does not require labeled data during training, it learns the structure of "normal" transactions and flags outliers.

This is particularly useful in fraud detection because:
- Fraudulent patterns evolve over time (concept drift)
- New fraud types may not exist in historical labeled data
- It can serve as a complementary model alongside supervised classifiers

We train it only on legitimate transactions and evaluate its ability to flag fraud on the test set.

In [11]:
# Train Isolation Forest on legitimate transactions only
scaler_if = StandardScaler()
X_train_scaled = scaler_if.fit_transform(X_train)
X_test_scaled = scaler_if.transform(X_test)

# Train only on non-fraud samples
X_train_legit = X_train_scaled[y_train == 0]

# contamination is set to the approximate fraud ratio in the dataset
fraud_ratio = y_train.mean()

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=fraud_ratio,
    random_state=random_state,
    n_jobs=-1
)

iso_forest.fit(X_train_legit)

# Predict on test set: -1 = anomaly (fraud), 1 = normal
y_pred_if = iso_forest.predict(X_test_scaled)

# Convert: -1 -> 1 (fraud), 1 -> 0 (legit)
y_pred_if = np.where(y_pred_if == -1, 1, 0)

# Anomaly scores (lower = more anomalous)
anomaly_scores = iso_forest.decision_function(X_test_scaled)
# Invert so higher score = more likely fraud (for AUPRC/ROC compatibility)
fraud_scores = -anomaly_scores

# Evaluate
if_metrics = {
    "Model": "Isolation_Forest",
    "Strategy": "unsupervised",
    "AUPRC": average_precision_score(y_test, fraud_scores),
    "ROC_AUC": roc_auc_score(y_test, fraud_scores),
    "Precision": precision_score(y_test, y_pred_if),
    "Recall": recall_score(y_test, y_pred_if),
    "F1": f1_score(y_test, y_pred_if),
    "Threshold": "auto (contamination)",
    "CM_TN": confusion_matrix(y_test, y_pred_if)[0, 0],
    "CM_FP": confusion_matrix(y_test, y_pred_if)[0, 1],
    "CM_FN": confusion_matrix(y_test, y_pred_if)[1, 0],
    "CM_TP": confusion_matrix(y_test, y_pred_if)[1, 1]
}

print("===== Isolation Forest Results =====")
print(f"AUPRC     : {if_metrics['AUPRC']:.6f}")
print(f"ROC AUC   : {if_metrics['ROC_AUC']:.6f}")
print(f"Precision : {if_metrics['Precision']:.4f}")
print(f"Recall    : {if_metrics['Recall']:.4f}")
print(f"F1 Score  : {if_metrics['F1']:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_if))

===== Isolation Forest Results =====
AUPRC     : 0.119612
ROC AUC   : 0.956531
Precision : 0.1909
Recall    : 0.2211
F1 Score  : 0.2049

Confusion Matrix:
[[56562    89]
 [   74    21]]


In [12]:
# Append Isolation Forest results to the comparison table
if_row = {k: v for k, v in if_metrics.items() if k != "Threshold"}
if_row["Threshold"] = np.nan
results_df = pd.concat([results_df, pd.DataFrame([if_row])], ignore_index=True)
results_df = results_df.sort_values("AUPRC", ascending=False).reset_index(drop=True)

print("===== UPDATED RANKING (all models) =====")
print(results_df[["Strategy", "Model", "AUPRC", "ROC_AUC", "Precision", "Recall", "F1"]].to_string(index=True))

# Save updated results
results_df.to_csv("../results/model_comparison_results.csv", index=False)
print("\nSaved: updated model_comparison_results.csv")

===== UPDATED RANKING (all models) =====
       Strategy                Model     AUPRC   ROC_AUC  Precision    Recall        F1
0         smote        Random_Forest  0.868279  0.977321   0.918605  0.831579  0.872928
1      baseline        Random_Forest  0.868078  0.961125   0.925000  0.778947  0.845714
2         under        Random_Forest  0.849415  0.974439   0.870588  0.778947  0.822222
3         smote  Logistic_Regression  0.765069  0.976855   0.793478  0.768421  0.780749
4      baseline  Logistic_Regression  0.763118  0.983728   0.784946  0.768421  0.776596
5         under  Logistic_Regression  0.684932  0.980625   0.682243  0.768421  0.722772
6  unsupervised     Isolation_Forest  0.119612  0.956531   0.190909  0.221053  0.204878
7         smote         Naives_Bayes  0.097004  0.958766   0.109940  0.768421  0.192358
8      baseline         Naives_Bayes  0.092209  0.967069   0.103933  0.778947  0.183395
9         under         Naives_Bayes  0.091863  0.967451   0.104079  0.778947  

Isolation Forest provides a complementary approach to supervised models. While its precision/recall may differ from tuned supervised classifiers, it has a key advantage: it can detect **novel fraud patterns** that were not present in the training labels.

In production, a common strategy is to use both:
- A supervised model (e.g., Random Forest) for known fraud patterns
- An unsupervised model (Isolation Forest) as an anomaly detector for emerging threats

# Conclusion

Several machine learning models were evaluated using different sampling strategies, along with an unsupervised anomaly detection approach (Isolation Forest).

The best performing supervised configuration was selected based on the **Area Under the Precision–Recall Curve (AUPRC)**, which is particularly suitable for highly imbalanced classification problems.

Isolation Forest was evaluated as a complementary unsupervised approach, capable of detecting novel fraud patterns without relying on labeled data.

The selected supervised model was saved for deployment in the API. In production, combining both supervised and unsupervised approaches provides a more robust fraud detection system.